# 99. Slice management

Lease renewal, SSH access, network repair and deletion. Run only the cells you need.

In [ ]:
from fabrictestbed_extensions.fablib.fablib import FablibManager as fablib_manager

import deployment_config as cfg
import fabric_helpers as fh
from fabric_helpers import run, run_script

fablib = fablib_manager()
slice_ = fh.get_slice(fablib)
nodes = fh.nodes_by_name(slice_)
control = nodes[cfg.CONTROL_NODE]
dynamos = nodes[cfg.DYNAMOS_NODE]
print(f"Slice {slice_.get_name()}: {slice_.get_state()}, lease end {slice_.get_lease_end()}")

## Renew the lease

In [ ]:
import datetime

RENEW_DAYS = 14
end = datetime.datetime.now(datetime.timezone.utc) + datetime.timedelta(days=RENEW_DAYS)
slice_.renew(end.strftime("%Y-%m-%d %H:%M:%S %z"))
print(f"New lease end: {fh.get_slice(fablib).get_lease_end()}")

## SSH commands (use with the `ssh_config` generated by 00_configure_fablib)

In [ ]:
fh.print_ssh_commands(slice_)

## Cluster status

In [ ]:
print(run(control, "kubectl get nodes -o wide", quiet=True))
print(run(control, "helm list --all-namespaces", quiet=True))
print(run(control, f"cat {cfg.INSTALLED_METHOD_FILE} 2>/dev/null || echo 'no method installed'", quiet=True))

## Repair networking after a VM reboot

Data-plane addresses and routes are not persistent. After a reboot, reapply them and turn swap off again.

In [ ]:
fh.configure_networking(slice_)
for node in slice_.get_nodes():
    run(node, "sudo swapoff -a", quiet=True)

## Delete the slice (irreversible)

In [ ]:
CONFIRM_DELETE = False

if CONFIRM_DELETE:
    slice_.delete()
    print(f"Deleted {cfg.SLICE_NAME}")
else:
    print("Skipped (CONFIRM_DELETE = False).")

In [ ]:
# should fail although it seems that I need to run twice the previous cell 
slice = fablib.get_slice(name=fablib)
slice.get_nodes()